In [ ]:
! pip install -q accelerate peft bitsandbytes transformers trl

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)

from peft import LoraConfig, PeftModel
from trl import SFTTrainer


In [ ]:

ds = load_dataset("nihal-mp/erkamd-human_assistant-llama2-1k")

README.md:   0%|          | 0.00/273 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  646kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
model_name = "Qwen/Qwen3.8-27B"

dataset_name = "nihal-mp/erkamd-human_assistant-llama2-1k"
new_model = "qwen-custom"

lora_r = 64
lora_alpha = 16
lora_dropout = 0.1

use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False


In [ ]:
output_dir = "./results"

num_train_epochs = 1
fp16 = False
bf16 = True
per_device_train_batch_size = 4
per_device_eval_batch_size = 4

gradient_accumulation_steps = 1

gradient_checkpointing = True

max_grad_norm = 0.3

learning_rate = 2e-4

optim = "paged_adamw_32bit"

lr_scheduler_type = "cosine"

max_steps = -1

warmup_ratio = 0.03

group_by_length = True

save_steps = 0

logging_steps = 25

max_seq_length = None
packing = False

device_map = {"" : 0}

In [ ]:

import os
import re
import inspect
import torch
import transformers
import trl

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer



model_name = "Qwen/Qwen3-8B"

dataset = load_dataset(dataset_name, split="train")

print("Original columns:", dataset.column_names)
print("Original example:", dataset[0])


def convert_example(example):
    text = example["text"].strip()

    match = re.fullmatch(
        r"\s*<s>\s*\[INST\]\s*(.*?)\s*\[/INST\]\s*(.*?)\s*</s>\s*",
        text,
        flags=re.DOTALL,
    )

    if match is None:
        raise ValueError(
            f"Unexpected dataset format: {text[:200]}"
        )

    question = re.sub(
        r"^Human:\s*", "", match.group(1).strip()
    )

    answer = re.sub(
        r"^Assistant:\s*", "", match.group(2).strip()
    )

    return {
        "prompt": [
            {"role": "user", "content": question}
        ],
        "completion": [
            {"role": "assistant", "content": answer}
        ],
    }


if "text" in dataset.column_names:
    dataset = dataset.map(
        convert_example,
        remove_columns=dataset.column_names,
        desc="Converting Llama 2 data to Qwen conversations",
    )
elif not (
    "prompt" in dataset.column_names
    and "completion" in dataset.column_names
):
    raise ValueError(
        f"Unsupported dataset columns: {dataset.column_names}"
    )

print("\nConverted example:")
print(dataset[0])


if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

if torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16

print("\nGPU:", torch.cuda.get_device_name(0))
print("Compute dtype:", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

model_load_kwargs = {
    "quantization_config": bnb_config if use_4bit else None,
    "device_map": device_map,
}


model_load_params = inspect.signature(
    AutoModelForCausalLM.from_pretrained
).parameters

if "dtype" in model_load_params or "kwargs" in model_load_params:
    model_load_kwargs["dtype"] = compute_dtype
else:
    model_load_kwargs["torch_dtype"] = compute_dtype

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_load_kwargs,
)

model.config.use_cache = False

if use_4bit:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=gradient_checkpointing,
    )

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"



peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


print("\nTRL version:", trl.__version__)
print("Transformers version:", transformers.__version__)

config_params = inspect.signature(SFTConfig).parameters

config_kwargs = {
    "output_dir": output_dir,
    "num_train_epochs": num_train_epochs,
    "per_device_train_batch_size": per_device_train_batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "gradient_checkpointing": gradient_checkpointing,
    "optim": optim,
    "save_strategy": "epoch",
    "logging_steps": logging_steps,
    "logging_strategy": "steps",
    "report_to": "tensorboard",
    "learning_rate": learning_rate,
    "max_grad_norm": max_grad_norm,
    "max_steps": max_steps,
    "lr_scheduler_type": lr_scheduler_type,
    "warmup_steps": 10,
    "fp16": compute_dtype == torch.float16,
    "bf16": compute_dtype == torch.bfloat16,
    "packing": False,
}

sequence_length = max_seq_length or 1024

if "max_length" in config_params:
    config_kwargs["max_length"] = sequence_length
elif "max_seq_length" in config_params:
    config_kwargs["max_seq_length"] = sequence_length


if "completion_only_loss" in config_params:
    config_kwargs["completion_only_loss"] = True


if "eos_token" in config_params:
    config_kwargs["eos_token"] = "<|im_end|>"


if "group_by_length" in config_params:
    config_kwargs["group_by_length"] = group_by_length

if "remove_unused_columns" in config_params:
    config_kwargs["remove_unused_columns"] = True

unsupported = [
    key for key in config_kwargs
    if key not in config_params
]

if unsupported:
    raise RuntimeError(
        "Your installed SFTConfig does not support: "
        + ", ".join(unsupported)
        + ". Check TRL/Transformers version compatibility."
    )

training_arguments = SFTConfig(**config_kwargs)



trainer_params = inspect.signature(SFTTrainer).parameters

trainer_kwargs = {
    "model": model,
    "train_dataset": dataset,
    "peft_config": peft_config,
    "args": training_arguments,
}

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    raise RuntimeError(
        "Unsupported SFTTrainer tokenizer API."
    )

trainer = SFTTrainer(**trainer_kwargs)

print("\nTrainable parameters:")
trainer.model.print_trainable_parameters()

print("\nStarting training...")
trainer.train()



save_path = os.path.join(output_dir, "final")

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("\nTraining completed!")
print(f"Fine-tuned adapter saved to: {save_path}")


Original columns: ['text']
Original example: {'text': '<s>[INST] Human: Is D-bifunctional protein deficiency inherited ? [/INST] Assistant: This condition is inherited in an autosomal recessive pattern, which means both copies of the gene in each cell have mutations. The parents of an individual with an autosomal recessive condition each carry one copy of the mutated gene, but they typically do not show signs and symptoms of the condition. </s>'}


Converting Llama 2 data to Qwen conversations:   0%|          | 0/1000 [00:00<?, ? examples/s]


Converted example:
{'prompt': [{'content': 'Is D-bifunctional protein deficiency inherited ?', 'role': 'user'}], 'completion': [{'content': 'This condition is inherited in an autosomal recessive pattern, which means both copies of the gene in each cell have mutations. The parents of an individual with an autosomal recessive condition each carry one copy of the mutated gene, but they typically do not show signs and symptoms of the condition.', 'role': 'assistant'}]}

GPU: NVIDIA A100-SXM4-80GB
Compute dtype: torch.bfloat16


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

EOS token: <|im_end|>
Pad token: <|endoftext|>

TRL version: 1.16.0
Transformers version: 5.20.2


...

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]


Trainable parameters:
trainable params: 174,587,904 || all params: 8,365,323,264 || trainable%: 2.0870

Starting training...


Step,Training Loss
25,1.469274


Step,Training Loss
25,1.469274
50,1.167104
75,1.090209
100,1.138405
125,0.939785
150,1.090730
175,0.960897
200,1.170717
225,1.058841
250,1.015069



Training completed!
Fine-tuned adapter saved to: ./results/final


In [ ]:

import torch


inference_model = trainer.model


inference_model.eval()

if hasattr(inference_model, "gradient_checkpointing_disable"):
    inference_model.gradient_checkpointing_disable()

inference_model.config.use_cache = True

prompt = "What model are u"

messages = [
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
)

inputs = {
    key: value.to(inference_model.device)
    for key, value in inputs.items()
}

with torch.inference_mode():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        use_cache=True,
    )

input_length = inputs["input_ids"].shape[-1]

generated_tokens = outputs[0][input_length:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("MODEL RESPONSE:\n")
print(response)


MODEL RESPONSE:

I am Qwen, a large-scale language model developed by Alibaba Group. I can answer questions, create text, translate, and more.
